In [1]:
from __future__ import annotations

import copy
import pickle
import numpy as np
from typing import Sequence, Optional

from pymatgen.core import Structure

In [2]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent.parent.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT)) 

from muonscripts.constants import constants

from muonscripts.muesr_tools.local_fields import multisite_pfields
from muonscripts.muesr_tools.utils import check_site_distances, get_atom_kinds
from muonscripts.muesr_tools.sample_muon_sites import anion_sites, target_sites

from muonscripts.muesr_tools.bayesian import BayesianMomentEstimator

GAMMA_MU = constants.MUON_GYROMAGNETIC_RATIO/constants.TWOPI
GAMMA_MU *=1e-6 # MHz/T 

In [4]:
def calc_multisite_fields(
    st: Structure,
    magmoms: np.ndarray,
    muon_positions: np.ndarray,
    k: Optional[Sequence[float]] = None,
    cont_field: float = 0.0,
    sphere_radius: int = 100
) -> np.ndarray:
    """
    Compute precession frequency per unit moment \nu/\mu (MHz / \mu_B).
    """
    result = multisite_pfields(
        structure=st.copy(),  # Fixed structure argument reference
        magmoms=magmoms,
        muon_positions=muon_positions,
        sphere_r=sphere_radius,
        k=k,
        cont_field=cont_field
    )

    # # Total Dipolar + Lorentz field in Tesla per \mu_B
    # B = result.dipolar + result.lorentz * lorentz_factor
    # B_norm = np.linalg.norm(B, axis=1) 

    # # Convert Tesla -> MHz i.e (Tesla /mu_B -> MHz /mu_B)
    # nu = B_norm * GAMMA_MU

    return result

In [5]:
import os
dpath='./'

In [6]:
filename = 'BNOO.cif'
filename = os.path.join(dpath, filename)
p_st = Structure.from_file(filename)
p_st
print(p_st)

Full Formula (Ba8 Na4 Os4 O24)
Reduced Formula: Ba2NaOsO6
abc   :   8.287000   8.287000   8.287000
angles:  90.000000  90.000000  90.000000
pbc   :       True       True       True
Sites (40)
  #  SP         a       b       c
---  ----  ------  ------  ------
  0  Ba    0.25    0.25    0.25
  1  Ba    0.75    0.25    0.25
  2  Ba    0.75    0.75    0.25
  3  Ba    0.25    0.75    0.25
  4  Ba    0.25    0.75    0.75
  5  Ba    0.25    0.25    0.75
  6  Ba    0.75    0.25    0.75
  7  Ba    0.75    0.75    0.75
  8  Na    0.5     0.5     0.5
  9  Na    0.5     0       0
 10  Na    0       0.5     0
 11  Na    0       0       0.5
 12  Os    0       0       0
 13  Os    0       0.5     0.5
 14  Os    0.5     0       0.5
 15  Os    0.5     0.5     0
 16  O     0.2256  0       0
 17  O     0       0.2256  0
 18  O     0.7744  0       0
 19  O     0       0.7744  0
 20  O     0       0       0.2256
 21  O     0       0       0.7744
 22  O     0.2256  0.5     0.5
 23  O     0       0.7256  0.

In [7]:
# Identify atom kinds

atm_kinds = get_atom_kinds(p_st)

print("\nAtom kinds:")
for kind, indices in atm_kinds.items():
    print(f"{kind}: {indices}")


print("\nOs indices:")
print(atm_kinds["Os"])


Atom kinds:
Ba: [0, 1, 2, 3, 4, 5, 6, 7]
Na: [8, 9, 10, 11]
Os: [12, 13, 14, 15]
O: [16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39]

Os indices:
[12, 13, 14, 15]


In [8]:
# Inspect Os positions
#
# IMPORTANT:
# This lets us verify the AFM assignment later.

print("\nOs fractional coordinates:")

for i in atm_kinds["Os"]:
    print(
        f"index = {i:3d}   "
        f"frac = {p_st.frac_coords[i]}"
    )


Os fractional coordinates:
index =  12   frac = [0. 0. 0.]
index =  13   frac = [0.  0.5 0.5]
index =  14   frac = [0.5 0.  0.5]
index =  15   frac = [0.5 0.5 0. ]


In [9]:
# Magnetic moment directions
#
# All moments have magnitude 1.
# Therefore the resulting frequency distribution is nu / mu.

dir_111 = np.array([1.0/np.sqrt(3), 1.0/np.sqrt(3), 1.0/np.sqrt(3)])

dir_001 = np.array([0.0, 0.0, 1.0])

print("\nMoment magnitudes:")
print("|dir_111| =", np.linalg.norm(dir_111))
print("|dir_001| =", np.linalg.norm(dir_001))


Moment magnitudes:
|dir_111| = 1.0
|dir_001| = 1.0


In [10]:
# AFM signs used in the present reconstruction
#
# We should verify these against the actual magnetic structure
# once the Os positions / paper figure are checked.

afm_signs = np.array(
    [-1.0, 1.0, 1.0, -1.0]
)

if len(atm_kinds["Os"]) != len(afm_signs):
    raise ValueError(
        "Number of Os sites does not match "
        "the number of AFM signs."
    )


In [11]:
# Construct magnetic structures

magmoms_FM111  = np.zeros((len(p_st), 3))
magmoms_AFM111 = np.zeros_like(magmoms_FM111)
magmoms_AFM001 = np.zeros_like(magmoms_FM111)


# FM [111]
magmoms_FM111[atm_kinds["Os"]] = dir_111

# AFM [111]
magmoms_AFM111[atm_kinds["Os"]] = afm_signs[:, None]* dir_111

# AFM [001]
magmoms_AFM001[atm_kinds["Os"]] = afm_signs[:, None]* dir_001

In [12]:
# Check magnetic moments

for label, moments in {
    "FM111": magmoms_FM111,
    "AFM111": magmoms_AFM111,
    "AFM001": magmoms_AFM001,
}.items():

    print(f"\n{label}")
    print("-" * 55)

    for i in atm_kinds["Os"]:
        m = moments[i]
        norm = np.linalg.norm(m)

        # Print 3D vector [mx, my, mz] with aligned signs
        if len(m) == 3:
            vec_str = f"[{m[0]:+8.4f}, {m[1]:+8.4f}, {m[2]:+8.4f}]"
        # Fallback for scalar/collinear moments
        else:
            vec_str = f"{m[0]:+8.4f}"

        print(f"Os {i:3d}: {vec_str}  |m| = {norm:7.4f}")


FM111
-------------------------------------------------------
Os  12: [ +0.5774,  +0.5774,  +0.5774]  |m| =  1.0000
Os  13: [ +0.5774,  +0.5774,  +0.5774]  |m| =  1.0000
Os  14: [ +0.5774,  +0.5774,  +0.5774]  |m| =  1.0000
Os  15: [ +0.5774,  +0.5774,  +0.5774]  |m| =  1.0000

AFM111
-------------------------------------------------------
Os  12: [ -0.5774,  -0.5774,  -0.5774]  |m| =  1.0000
Os  13: [ +0.5774,  +0.5774,  +0.5774]  |m| =  1.0000
Os  14: [ +0.5774,  +0.5774,  +0.5774]  |m| =  1.0000
Os  15: [ -0.5774,  -0.5774,  -0.5774]  |m| =  1.0000

AFM001
-------------------------------------------------------
Os  12: [ -0.0000,  -0.0000,  -1.0000]  |m| =  1.0000
Os  13: [ +0.0000,  +0.0000,  +1.0000]  |m| =  1.0000
Os  14: [ +0.0000,  +0.0000,  +1.0000]  |m| =  1.0000
Os  15: [ -0.0000,  -0.0000,  -1.0000]  |m| =  1.0000


sample sites and compute freqs/muB for each magnetic configurations

In [13]:
min_cation_distances={
    "Ba": 1.0,
    "Na": 1.0,
    "Os": 1.0,
} # new

seed_no = 42
n_samples = 20000
O_distance = (0.9, 1.10)

candidate_sites = anion_sites(
    p_st.copy(),
    n_samples=n_samples,

    anion_specie=("O",),
    anion_distance=O_distance,

    min_cation_distances=min_cation_distances,

    batch_factor=50,

    seed=seed_no,
)

print(
    f"\nGenerated "
    f"{len(candidate_sites)} "
    f"candidate muon sites."
)


Generated 20000 candidate muon sites.


In [14]:
# Add positions to original structure
p_stc = p_st.copy()
for pos in candidate_sites:
    p_stc.append('H', pos)

file='BNOO_randomsamples_muonsites.cif'
file = os.path.join(dpath, file)

p_stc = p_stc.copy()
p_stc = Structure(
    p_stc.lattice, 
    p_stc.species, 
    p_stc.frac_coords,
    site_properties=p_stc.site_properties,   
)

p_stc.to(file)
print(
    f"Saved candidate sites to "
    f"{file}"
)

Saved candidate sites to ./BNOO_randomsamples_muonsites.cif


In [15]:
distances = check_site_distances(
    p_st,
    candidate_sites,
)


Site-distance diagnostics:
  Ba: min = 1.8404 Å, max = 3.0241 Å, mean = 2.3009 Å, median = 2.2734 Å
  Na: min = 1.1766 Å, max = 3.2962 Å, mean = 2.3913 Å, median = 2.4473 Å
  Os: min = 1.0000 Å, max = 2.9680 Å, mean = 2.0926 Å, median = 2.1584 Å
   O: min = 0.9000 Å, max = 1.1000 Å, mean = 1.0060 Å, median = 1.0093 Å


check how long to compute freqs

UNCOMMENT LINES

In [16]:
# import time

# for n in [10, 50, 100, 200]:

#     sites = candidate_sites[:n]

#     t0 = time.perf_counter()

#     result = multisite_pfields(
#         structure=p_st.copy(),
#         magmoms=magmoms_FM111,
#         muon_positions=sites,
#         sphere_r=100,
#     )

#     elapsed = time.perf_counter() - t0

#     print(
#         f"{n:6d} muons : "
#         f"{elapsed:8.2f} s  "
#         f"({elapsed / n:.4f} s/muon)"
#     )

# result.dipolar_norm
# result.lorentz_norm * GAMMA_MU

UNCOMMENT LINES BELOW:

Calculate nu / mu

In [17]:
sphere_radius=80
lorentz_factor=0

In [18]:
# Test with FM

result = calc_multisite_fields(
    st=p_st.copy(),
    magmoms=magmoms_FM111,
    muon_positions=candidate_sites[0:2],
    sphere_radius=sphere_radius
)

# result contains field contributions in Tesla /mu_B
# result.total        is total field
# result.dipolar      is dipolar contributions
# result.lorentz      is lorentz contributions
# result.dipolar_tot  is dipolar+lorentz contributions
# result.contact      is contact contributions
# *_norm              are the magnitudes

Bdip = result.dipolar
Bdip_norm = np.linalg.norm(Bdip, axis=1)
# # Convert Tesla -> MHz i.e (Tesla /mu_B -> MHz /mu_B)
nu = Bdip_norm * GAMMA_MU
Bdip, Bdip_norm, nu

(array([[ 0.01722098, -0.04290657, -0.01981827],
        [ 0.13069658,  0.73167722,  0.30966703]]),
 array([0.05030209, 0.80518744]),
 array([  6.81788477, 109.13414774]))

In [19]:
print("\nCalculating FM [111]...")

results_FM111 = calc_multisite_fields(
    st=p_st.copy(),
    magmoms=magmoms_FM111,
    muon_positions=candidate_sites,
    sphere_radius=sphere_radius
)


Calculating FM [111]...


In [20]:
print("\nCalculating AFM [111]...")

results_AFM111 = calc_multisite_fields(
    st=p_st.copy(),
    magmoms=magmoms_AFM111,
    muon_positions=candidate_sites,
    sphere_radius=sphere_radius
)


Calculating AFM [111]...


In [21]:
print("\nCalculating AFM [001]...")

results_AFM001 = calc_multisite_fields(
    st=p_st.copy(),
    magmoms=magmoms_AFM001,
    muon_positions=candidate_sites,
    sphere_radius=sphere_radius
)


Calculating AFM [001]...


In [26]:
# Field diagnostics
field_contributions = {
    "FM111": results_FM111,
    "AFM111": results_AFM111,
    "AFM001": results_AFM001,
}

# Dictionary to store structured results per label
fields_diagnostics = {}

for label, res in field_contributions.items():
    Bdip = res.dipolar
    Bdip_norm = np.linalg.norm(Bdip, axis=1)
    
    # Convert Tesla -> MHz i.e (Tesla /mu_B -> MHz /mu_B)
    nu_values = Bdip_norm * GAMMA_MU

    # Store in nested dictionary
    fields_diagnostics[label] = {
        "results": res,
        "Bdip": Bdip,
        "Bdip_norm": Bdip_norm,
        "nu_values": nu_values,
        "stats": {
            "min": nu_values.min(),
            "max": nu_values.max(),
            "mean": nu_values.mean(),
            "median": np.median(nu_values),
            "std": nu_values.std(),
        },
    }

    # Print summary
    print(f"\n{label}")
    print("-" * 25)
    for stat_name, stat_val in fields_diagnostics[label]["stats"].items():
        print(f"  {stat_name:<6s} = {stat_val:.6f}")


FM111
-------------------------
  min    = 4.186928
  max    = 225.709377
  mean   = 30.678359
  median = 16.979722
  std    = 34.328659

AFM111
-------------------------
  min    = 7.159365
  max    = 226.755625
  mean   = 31.956401
  median = 17.694584
  std    = 33.581202

AFM001
-------------------------
  min    = 11.942353
  max    = 244.003471
  mean   = 33.338448
  median = 18.747270
  std    = 32.554612


saved results:

In [ ]:
BNOO_freqs = {
    # "FM111":  results_FM111,
    # "AFM111": results_AFM111,
    # "AFM001": results_AFM001,

    "fields": field_contributions,
    "fields_diagnostics": fields_diagnostics,

    "sample_sites": candidate_sites,

    "host_lattice": p_st.copy(),

    "structure_sites": p_stc,

    # Store parameters so we know exactly how the data were generated.
    "parameters": {
        "n_samples": n_samples,
        "O_distance": O_distance,
        "min_cation_distances": min_cation_distances,
        "sphere_radius": sphere_radius,
        "seed": seed_no,
        "lorentz_factor": lorentz_factor,
        'gamma_mu': GAMMA_MU
    },
}

filename = "BNOO_freqs.pkl"
filename = os.path.join(dpath, filename)
with open(filename, "wb") as f:
    pickle.dump(BNOO_freqs, f)

print(
    f"\nSaved frequency distributions to "
    f"{filename}"
)


Saved frequency distributions to ./BNOO_freqs.pkl


load results:

In [25]:
# Load the dictionary back into memory in binary read mode ('rb')
filename = "BNOO_freqs.pkl"
filename = os.path.join(dpath, filename)
with open(filename, "rb") as f:
    loaded_BNOO_freqs = pickle.load(f)

loaded_BNOO_freqs.keys()

dict_keys(['fields', 'sample_sites', 'host_lattice', 'structure_sites', 'parameters'])